# Tine: hidden-state representations for the index model

Use a **GPU** runtime (L4 or A100; T4 is too small for a 7B model in half precision with long prompts).
Run all cells. About 15 GB of model weights are downloaded, then roughly 20-40 minutes of compute.

What it does: reads each of the 1,555 coding prompts once with a frozen Qwen2.5-7B-Instruct, up to
layer 21, and saves the vector at the last position. Nothing is trained and no text is generated.
These vectors replace the topic embeddings in front of the kNN router, following arXiv:2610.19198.
See `docs/INDEX_MODEL.md` in the repository.

Needs `MyDrive/tine/coding-lcb-swe.jsonl` from the earlier notebooks. Output goes to `MyDrive/tine/`.

In [ ]:
# 1. Mount Drive, get the repository, check the GPU and the input.
from google.colab import drive
drive.mount("/content/drive")

import json, os, subprocess, sys
from pathlib import Path

BRANCH = "claude/grade-dispute"  # change to "main" once this branch is merged
ROOT = Path("/content/drive/MyDrive/tine")
REPO = Path("/content/tine")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                    "https://github.com/Nandu-Dammalapti/tine.git", str(REPO)], check=True)
os.chdir(REPO)
print("tine revision:", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "NO GPU: change the runtime type")
INPUT = ROOT / "coding-lcb-swe.jsonl"
assert INPUT.exists(), f"Missing {INPUT}: run the step 2 notebook first"
CACHE = "/content/hf"  # model weights stay on the runtime disk, not on Drive

In [ ]:
# 2. Smoke test on 16 prompts (downloads the model: several minutes the first time).
SMOKE = ROOT / "lcb-swe-qwen25-7b-l21-issue-smoke.npz"
subprocess.run([sys.executable, "scripts/hidden_states.py", "--input", str(INPUT), "--output", str(SMOKE),
                "--prompt-view", "issue", "--limit", "16", "--batch-size", "4", "--cache-dir", CACHE], check=True)
print(json.loads(Path(str(SMOKE) + ".json").read_text()))

In [ ]:
# 3. All 1,555 prompts. Resumable: rerun cells 1 and 3 after a disconnect.
OUTPUT = ROOT / "lcb-swe-qwen25-7b-l21-issue-2048.npz"
subprocess.run([sys.executable, "scripts/hidden_states.py", "--input", str(INPUT), "--output", str(OUTPUT),
                "--prompt-view", "issue", "--batch-size", "4", "--cache-dir", CACHE], check=True)

In [ ]:
# 4. Check the result. Copy this output back to Claude.
import numpy as np
with np.load(OUTPUT, allow_pickle=False) as saved:
    print("vectors:", saved["embeddings"].shape, saved["embeddings"].dtype)
    print("first and last task:", saved["task_ids"][0], saved["task_ids"][-1])
print(json.loads(Path(str(OUTPUT) + ".json").read_text()))
print("saved:", OUTPUT)

Done. Choose **Runtime → Disconnect and delete runtime**. The vectors are on Drive at
`MyDrive/tine/lcb-swe-qwen25-7b-l21-issue-2048.npz`; the router evaluation runs from there on a CPU.